In [ ]:
%reset -f 
import numpy as np
import matplotlib.pyplot as plt

from starwinds_readplt.dataset import Dataset

from batcamp import OctreeInterpolator

# 

In [ ]:
import pooch
url="https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz"
known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136"
members = [
    "run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt",
    "run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt",
]

files = pooch.retrieve(url, 
                       known_hash=known_hash, 
                       progressbar=True,                       
                       processor=pooch.Untar(members=members)
)

print(files)

In [ ]:
# Read the dataset
ids = ("SC", "IH")
ds = {ids: Dataset.from_file(file) for ids, file in zip(ids, files)}
print("".join([f"{c}:\n{d}\n" for c, d in ds.items()]))


In [ ]:

# SC file is spherical/rpa
interp = {ids: OctreeInterpolator(ds, ["Rho [g/cm^3]"]) for ids, ds in ds.items()}
print("".join([f"{c}:\n{i}\n" for c, i in interp.items()]))


In [ ]:
# Cartesian slice
X, Y = np.meshgrid(np.linspace(-60, 60, 512), np.linspace(-60, 60, 512))
Z = np.zeros_like(X)
R = np.sqrt(X**2 + Y**2 + Z**2)

merge_radius = 50
rho = np.where(R <= merge_radius, interp["SC"](X, Y, Z), interp["IH"](X, Y, Z))

pcm = plt.pcolormesh(X, Y, rho, norm="log")
plt.colorbar(pcm)
plt.title("Merged SC/IH")
plt.show()

In [ ]:
# Cartesian slice with spherical grid
R = np.geomspace(1, 150, 1024)
AZ = np.linspace(0, 2 * np.pi, 512)
R, AZ = np.meshgrid(R, AZ)
X = R * np.cos(AZ)
Y = R * np.sin(AZ)
Z = np.zeros_like(X)

rho = np.where(R <= merge_radius, interp["SC"](X, Y, Z), interp["IH"](X, Y, Z))

fig, ax = plt.subplots()
pcm = ax.pcolormesh(X, Y, rho, norm="log")
plt.colorbar(pcm)
ax.set_title("Merged SC/IH (spherical grid)")
ax.set_aspect("equal")
plt.show()